# Metadata baseline model

In [1]:
import numpy as np
import pandas as pd

### Load split data

In [2]:
train_df = pd.read_csv(
    "data/train_df.csv",
    parse_dates=["publish_timestamp"]  # converts datestime string into datetime object
)
test_df = pd.read_csv(
    "data/test_df.csv",
    parse_dates=["publish_timestamp"]
)


In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1471 entries, 0 to 1470
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   user_id                   1471 non-null   int64         
 1   following                 1471 non-null   int64         
 2   publish_timestamp         1471 non-null   datetime64[ns]
 3   has_location              1471 non-null   int64         
 4   is_carousel               1471 non-null   int64         
 5   num_images                1471 non-null   int64         
 6   is_sponsored              1471 non-null   int64         
 7   image_path                1471 non-null   object        
 8   caption                   1443 non-null   object        
 9   follower_following_ratio  1471 non-null   float64       
 10  hour                      1471 non-null   int64         
 11  day                       1471 non-null   object        
 12  is_weekend          

### Define column groups to be used in metadata baseline model

In [4]:
numeric_cols = [
    "following",
    "follower_following_ratio",
    "is_weekend",
    "has_location",
    "is_carousel",
    "num_images",
    "is_sponsored",
    "caption_word_count",
    "num_hashtags"
]

categorical_cols = ["day", "hour"]


### Preprocessor

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)


In [6]:
# select input features from dataframe
X_train = train_df[numeric_cols + categorical_cols]
X_test  = test_df[numeric_cols + categorical_cols]

# apply preprocessing
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)


In [7]:
# check that column size increased due to encoding
# (number_of_samples, number_of_features_after_encoding)
X_train_proc.shape

(1471, 40)

In [9]:
# select target label from dataframe
y_train = train_df["engagement_label"].values
y_test  = test_df["engagement_label"].values

### Convert preprocessed features into PyTorch tensors

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader

class MetadataDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y) # num of samples in dataset

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx] # return sample at index


In [11]:
# Creating dataset objects
train_ds = MetadataDataset(X_train_proc, y_train)
test_ds  = MetadataDataset(X_test_proc, y_test)

# DataLoader splits data into batches
batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True) # randomize order every epoch
test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False) # predictable evaluation


In [12]:
import torch.nn as nn

class MetadataMLP(nn.Module):
    def __init__(self, input_dim): # input_dim is number of processed features
        super().__init__()          # call constructor of nn.Module
        self.net = nn.Sequential(   # stacks layers in order
            nn.Linear(input_dim, 64),   # Fully connected layer (40 features -> 64 hidden units)
            nn.ReLU(),                  # ReLU for non-linearity to learn complex patterns
            nn.Dropout(0.3),            # randomly drop 30% of neurons to prevent overfitting
            nn.Linear(64, 32),          # 2nd hidden layer, compresses information
            nn.ReLU(),
            nn.Linear(32, 3)            # output layer, 3 engagement classes, outputs logits.
        )

    def forward(self, x): # forward pass
        return self.net(x)


In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu" # use GPU if available

# Input size matching one-hot expanded features
input_dim = X_train_proc.shape[1]

# Instantiate model, move weights to device
model = MetadataMLP(input_dim).to(device)

# Set loss function and optimizer
criterion = nn.CrossEntropyLoss() # multiclass classification, labels are integers, logits are raw scores
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3) # learning rate 0.001 is standard

# training loop
def train_epoch(loader):
    # Put model in training mode
    model.train()

    # Initialize accumulator for total loss
    total_loss = 0

    # Loops over batches in training set
    for X, y in loader:
        # Moves x y to same device
        X, y = X.to(device), y.to(device)

        # Clear old gradients from previous batch
        optimizer.zero_grad()

        # Feeds batch into model, perform forward pass, output logits
        logits = model(X)

        # Compute loss for this batch
        loss = criterion(logits, y)

        # Performs backpropagation: computes gradients of the loss with respect to each model parameter.
        loss.backward()

        # Updates model parameters using those gradients
        optimizer.step()

        # Adds the numeric loss value (converted from a tensor with .item()) to the total loss accumulator.
        total_loss += loss.item()

    # Average loss = total loss / number of batches
    avg_loss = total_loss / len(loader)
    
    return avg_loss




In [14]:
def test_epoch(loader):
    # Set model to evaluation mode. Disables dropout, freezes batchnorm stats
    model.eval()

    # Initialize accumulator for total loss
    total_loss = 0

    with torch.no_grad(): # Disables gradient tracking inside this block.
    # This saves memory and time, since we don’t need gradients for validation.
    # It also ensures the model’s weights won’t accidentally be updated.
        for X, y in loader:
            X, y = X.to(device), y.to(device)
 
            logits = model(X)

            total_loss += criterion(logits, y).item()

    # Compute average loss
    avg_loss = total_loss / len(loader)
    
    return avg_loss

In [15]:
from sklearn.metrics import confusion_matrix, f1_score

def evaluate_metrics(loader, model):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)

            logits = model(X)
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    # Accuracy
    correct = sum(p == y for p, y in zip(all_preds, all_labels))
    accuracy = correct / len(all_labels)

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)

    # Macro F1
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return accuracy, cm, macro_f1


In [ ]:
for epoch in range(10):
    train_loss = train_epoch(train_loader)
    test_loss  = test_epoch(test_loader)

    # Metrics
    train_acc, train_cm, train_macro_f1 = evaluate_metrics(train_loader, model)
    test_acc, test_cm, test_macro_f1 = evaluate_metrics(test_loader, model)

    print(f"\nEpoch {epoch+1}")
    print(f"Train   | loss={train_loss:.4f}, acc={train_acc:.3f}, macro-F1={train_macro_f1:.3f}")
    print(f"Test    | loss={test_loss:.4f}, acc={test_acc:.3f}, macro-F1={test_macro_f1:.3f}")

    print("Train Confusion Matrix:")
    print(train_cm)

    print("Test Confusion Matrix:")
    print(test_cm)



Epoch 1
Train   | loss=1.0917, acc=0.470, macro-F1=0.453
Valid   | loss=1.0730, acc=0.452, macro-F1=0.446
Train Confusion Matrix:
[[133 149 180]
 [ 75 205 216]
 [ 49 110 354]]
Validation Confusion Matrix:
[[50 31 66]
 [18 43 53]
 [10 28 77]]

Epoch 2
Train   | loss=1.0522, acc=0.494, macro-F1=0.472
Valid   | loss=1.0230, acc=0.487, macro-F1=0.468
Train Confusion Matrix:
[[190  95 177]
 [105 143 248]
 [ 54  66 393]]
Validation Confusion Matrix:
[[64 25 58]
 [25 31 58]
 [14 13 88]]

Epoch 3
Train   | loss=0.9986, acc=0.516, macro-F1=0.491
Valid   | loss=0.9791, acc=0.505, macro-F1=0.483
Train Confusion Matrix:
[[238  87 137]
 [121 121 254]
 [ 61  52 400]]
Validation Confusion Matrix:
[[68 23 56]
 [24 30 60]
 [15  8 92]]

Epoch 4
Train   | loss=0.9654, acc=0.564, macro-F1=0.534
Valid   | loss=0.9348, acc=0.548, macro-F1=0.504
Train Confusion Matrix:
[[338  42  82]
 [168 115 213]
 [103  34 376]]
Validation Confusion Matrix:
[[95 15 37]
 [39 21 54]
 [17  8 90]]

Epoch 5
Train   | loss=0.93